# Chapter 1 Lab: Framing Problems and Building Baselines

This notebook demonstrates how problem framing shapes your ML pipeline. We'll build a spam detection problem from scratch, explore how different framings reveal different insights, and build baselines that anchor our expectations.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, confusion_matrix

%matplotlib inline

# Seed for reproducibility
np.random.seed(42)

## Section 1: Framing a Prediction Problem

Before we build any model, we must frame the problem: What are we predicting? Why? What tradeoffs matter?

Let's generate a toy email spam dataset and explore three different framings.

In [ ]:
# Generate a toy email dataset
n_samples = 200
np.random.seed(42)

# Spam emails tend to have more links and caps
spam_word_count = np.random.normal(loc=120, scale=30, size=n_samples // 2)
spam_has_link = np.random.binomial(n=1, p=0.7, size=n_samples // 2)
spam_caps_ratio = np.random.uniform(0.2, 0.8, size=n_samples // 2)

# Ham (legitimate) emails are more measured
ham_word_count = np.random.normal(loc=200, scale=50, size=n_samples // 2)
ham_has_link = np.random.binomial(n=1, p=0.2, size=n_samples // 2)
ham_caps_ratio = np.random.uniform(0.01, 0.15, size=n_samples // 2)

# Combine
X_word_count = np.concatenate([spam_word_count, ham_word_count])
X_has_link = np.concatenate([spam_has_link, ham_has_link])
X_caps_ratio = np.concatenate([spam_caps_ratio, ham_caps_ratio])
y = np.concatenate([np.ones(n_samples // 2), np.zeros(n_samples // 2)]).astype(int)

# Stack into feature matrix
X = np.column_stack([X_word_count, X_has_link, X_caps_ratio])

print(f"Dataset shape: {X.shape}")
print(f"Class distribution: {np.sum(y == 1)} spam, {np.sum(y == 0)} ham")
print(f"\nSpam class statistics:")
print(f"  word_count: mean={X[y==1, 0].mean():.1f}, std={X[y==1, 0].std():.1f}")
print(f"  has_link: {X[y==1, 1].sum() / np.sum(y==1):.1%}")
print(f"  caps_ratio: mean={X[y==1, 2].mean():.3f}, std={X[y==1, 2].std():.3f}")
print(f"\nHam class statistics:")
print(f"  word_count: mean={X[y==0, 0].mean():.1f}, std={X[y==0, 0].std():.1f}")
print(f"  has_link: {X[y==0, 1].sum() / np.sum(y==0):.1%}")
print(f"  caps_ratio: mean={X[y==0, 2].mean():.3f}, std={X[y==0, 2].std():.3f}")

## Section 2: How Framing Changes Evaluation

The same data can be framed three different ways, each revealing different insights:

1. **Accuracy-first**: "Minimize total misclassifications"
2. **Precision-first**: "Avoid false alarms—users trust our spam filter"
3. **Recall-first**: "Catch all spam, even if we misclassify some ham"

Let's evaluate a simple baseline under each framing.

In [ ]:
# Train a simple logistic regression on the full dataset (for demonstration)
model = LogisticRegression(random_state=42)
model.fit(X, y)
y_pred = model.predict(X)

# Compute metrics
acc = accuracy_score(y, y_pred)
prec = precision_score(y, y_pred)
rec = recall_score(y, y_pred)

print("Framing 1: Accuracy-First")
print(f"  Accuracy = {acc:.3f}")
print(f"  Interpretation: {acc*100:.1f}% of emails classified correctly")
print(f"  Question: Is this good enough?\n")

print("Framing 2: Precision-First")
print(f"  Precision = {prec:.3f}")
print(f"  Interpretation: {prec*100:.1f}% of flagged emails are actually spam")
print(f"  Why it matters: Users trust our spam folder; false alarms erode trust\n")

print("Framing 3: Recall-First")
print(f"  Recall = {rec:.3f}")
print(f"  Interpretation: We catch {rec*100:.1f}% of actual spam")
print(f"  Why it matters: Missing spam is worse than a false alarm\n")

# Show confusion matrix
tn, fp, fn, tp = confusion_matrix(y, y_pred).ravel()
print("Confusion Matrix:")
print(f"  True Negatives (correct ham): {tn}")
print(f"  False Positives (ham → spam): {fp}")
print(f"  False Negatives (spam → ham): {fn}")
print(f"  True Positives (correct spam): {tp}")

## Section 3: Majority-Class Baseline

The simplest baseline: always predict the most common class. This reveals why accuracy alone is misleading on imbalanced data.

In [ ]:
# Majority-class baseline: always predict the most common class
majority_class = np.argmax(np.bincount(y))
y_majority_baseline = np.full_like(y, majority_class)

baseline_acc = accuracy_score(y, y_majority_baseline)
baseline_prec = precision_score(y, y_majority_baseline, zero_division=0)
baseline_rec = recall_score(y, y_majority_baseline, zero_division=0)

print("MAJORITY-CLASS BASELINE")
print(f"Strategy: Always predict class {majority_class} (Ham)\n")
print(f"Accuracy: {baseline_acc:.3f}")
print(f"Precision: {baseline_prec:.3f}")
print(f"Recall: {baseline_rec:.3f}\n")

# Compare to logistic regression
print("LOGISTIC REGRESSION (same data, for comparison)")
print(f"Accuracy: {acc:.3f}")
print(f"Precision: {prec:.3f}")
print(f"Recall: {rec:.3f}\n")

print("KEY INSIGHT:")
print(f"The majority-class baseline achieves {baseline_acc:.1%} accuracy just by guessing 'ham'.")
print(f"But it catches {baseline_rec:.1%} of actual spam (terrible for a spam filter!).")
print(f"Accuracy alone is a misleading metric for imbalanced problems.")

## Section 4: Rule-Based Keyword Baseline

Before building a fancy ML model, try a simple rule: if has_link=1 AND caps_ratio > 0.3, predict spam.

In [ ]:
# Simple keyword rule: has_link AND high caps
y_rule_baseline = ((X[:, 1] == 1) & (X[:, 2] > 0.3)).astype(int)

rule_acc = accuracy_score(y, y_rule_baseline)
rule_prec = precision_score(y, y_rule_baseline, zero_division=0)
rule_rec = recall_score(y, y_rule_baseline, zero_division=0)

print("RULE-BASED BASELINE")
print(f"Strategy: Predict spam if (has_link == 1) AND (caps_ratio > 0.3)\n")
print(f"Accuracy: {rule_acc:.3f}")
print(f"Precision: {rule_prec:.3f}")
print(f"Recall: {rule_rec:.3f}\n")

# Create comparison table
print("BASELINE COMPARISON")
print("-" * 60)
print(f"{'Strategy':<30} {'Accuracy':<12} {'Precision':<12} {'Recall':<12}")
print("-" * 60)
print(f"{'Majority class':<30} {baseline_acc:<12.3f} {baseline_prec:<12.3f} {baseline_rec:<12.3f}")
print(f"{'Rule-based (has_link+caps)':<30} {rule_acc:<12.3f} {rule_prec:<12.3f} {rule_rec:<12.3f}")
print(f"{'Logistic Regression':<30} {acc:<12.3f} {prec:<12.3f} {rec:<12.3f}")
print("-" * 60)

## Section 5: Train/Test Split Discipline

The golden rule: evaluate only on held-out test data. Let's see what happens if we cheat by evaluating on training data.

In [ ]:
# Proper train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)

# Train logistic regression
model_proper = LogisticRegression(random_state=42)
model_proper.fit(X_train, y_train)

# Evaluate on training data (cheating)
y_train_pred = model_proper.predict(X_train)
train_acc = accuracy_score(y_train, y_train_pred)

# Evaluate on test data (proper)
y_test_pred = model_proper.predict(X_test)
test_acc = accuracy_score(y_test, y_test_pred)

print("TRAIN/TEST SPLIT DISCIPLINE")
print(f"Training set size: {len(X_train)}")
print(f"Test set size: {len(X_test)}\n")
print(f"Accuracy on TRAINING data (cheating): {train_acc:.3f}")
print(f"Accuracy on TEST data (honest): {test_acc:.3f}\n")
print(f"Difference (overfitting indicator): {(train_acc - test_acc):.3f}\n")

# Per-metric comparison
train_prec = precision_score(y_train, y_train_pred)
train_rec = recall_score(y_train, y_train_pred)
test_prec = precision_score(y_test, y_test_pred)
test_rec = recall_score(y_test, y_test_pred)

print("Detailed Comparison:")
print("-" * 60)
print(f"{'Metric':<15} {'Train':<15} {'Test':<15} {'Gap':<15}")
print("-" * 60)
print(f"{'Accuracy':<15} {train_acc:<15.3f} {test_acc:<15.3f} {(train_acc-test_acc):<15.3f}")
print(f"{'Precision':<15} {train_prec:<15.3f} {test_prec:<15.3f} {(train_prec-test_prec):<15.3f}")
print(f"{'Recall':<15} {train_rec:<15.3f} {test_rec:<15.3f} {(train_rec-test_rec):<15.3f}")
print("-" * 60)

## Section 6: Visualizing Train/Test Performance

In [ ]:
# Plot train vs test accuracy
fig, ax = plt.subplots(1, 1, figsize=(8, 5))

metrics = ['Accuracy', 'Precision', 'Recall']
train_scores = [train_acc, train_prec, train_rec]
test_scores = [test_acc, test_prec, test_rec]

x = np.arange(len(metrics))
width = 0.35

bars1 = ax.bar(x - width/2, train_scores, width, label='Train', color='#2E86AB', alpha=0.8)
bars2 = ax.bar(x + width/2, test_scores, width, label='Test', color='#A23B72', alpha=0.8)

ax.set_ylabel('Score', fontsize=12, fontweight='bold')
ax.set_title('Train vs Test Performance: Evaluating on Held-Out Data', fontsize=13, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(metrics, fontsize=11)
ax.set_ylim([0, 1.05])
ax.legend(fontsize=11, loc='upper right')
ax.grid(axis='y', alpha=0.3, linestyle='--')

# Add value labels on bars
for bars in [bars1, bars2]:
    for bar in bars:
        height = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2., height,
                f'{height:.2f}', ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.show()

print("The test scores are our honest estimates of how the model will perform in production.")

## What to Notice

1. **Framing is foundational**: The same data can be evaluated as accuracy, precision, or recall problems—each frames the solution differently.

2. **Baselines matter**: Before building fancy models, establish what a simple baseline achieves. The majority-class baseline is shockingly hard to beat on imbalanced data.

3. **Accuracy is misleading**: On a 50-50 balanced dataset, 50% accuracy is the floor. But on imbalanced data, a naive "always ham" strategy gets 50% without solving the problem.

4. **Train/test split is non-negotiable**: Evaluating on training data overstates performance. Always hold out a test set for honest evaluation.

5. **Rules vs learned models**: Simple rules (if has_link AND caps > 0.3) can be surprisingly effective and are interpretable. Don't jump to ML without trying rules first.